In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1fhhk3bX27gVe332cUCu7ViJG8FlpL7LQ", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/04_01_intro.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Notebook 4: Information Provenance & Scratchpads

**Claude Certified Architect Prep** | Pod 5: Context Management & Reliability

---

In Notebooks 1-3, we learned how to manage context windows, escalate to humans, and handle errors across multi-agent systems. But there is a more fundamental reliability question we have not yet addressed: **where did this information come from?**

When an AI agent tells you "The payment deadline is March 31, 2024," you need to know: Which document? Which page? Which paragraph? Was this extracted today, or six months ago from a document that has since been updated?

If you cannot answer these questions, you cannot trust the output. And if you cannot trust the output, you do not have a production system -- you have a liability.

In this notebook, you will build two critical reliability systems: **claim-source mappings** that trace every extracted fact back to its origin, and **scratchpad files** that preserve intermediate results across agent turns without exhausting the context window.

**What you will build:**
- A claim-source mapping system with document, page, and paragraph tracing
- A temporal metadata tracker that flags outdated information
- A scratchpad schema for multi-step analysis
- A mock multi-step pipeline that uses scratchpads to process 50 documents without context exhaustion

**Prerequisites:** Notebooks 1-3 (Context Management, Escalation, Error Propagation)

**Estimated time:** 55 minutes

[Ask the AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/context-management-reliability/practice/4/assistant)

In [ ]:
# ============================================================
# Setup: Install dependencies
# ============================================================
# Run this cell first. All packages are available on Colab by default.

!pip install -q matplotlib numpy

import json
import copy
import hashlib
import textwrap
import random
from typing import Any, Optional
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

random.seed(42)

# Utility: pretty-print JSON
def pp(obj: Any) -> None:
    """Pretty-print a Python object as formatted JSON."""
    print(json.dumps(obj, indent=2, default=str))

# Utility: simulate current time
NOW = datetime(2024, 3, 15, 14, 30, 0)

print("Setup complete.")
print(f"Simulated current time: {NOW.isoformat()}")
print("No API keys required -- this notebook uses mock data throughout.")

In [ ]:
#@title 🎧 Listen: Why Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_02_why_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 1: Why Does This Matter?

Consider this scenario. You are building a data extraction agent that processes 50 legal contracts simultaneously. It extracts parties, payment terms, deadlines, and obligations from each contract and produces a consolidated summary.

The agent outputs: *"The payment deadline is March 31, 2024."*

Now ask yourself:
- **Which of the 50 contracts** does this come from?
- **What page and paragraph** contains this deadline?
- **Was the contract amended** after this fact was extracted?
- **Did the agent actually read this in a document**, or did it hallucinate it?

Without provenance tracking, you have no way to answer any of these questions. The extracted "fact" is indistinguishable from a hallucination.

This matters for three reasons:

1. **Auditability** -- Regulators, legal teams, and compliance officers need to verify where facts came from.
2. **Hallucination prevention** -- If the agent cannot point to a specific source, the claim is flagged.
3. **Temporal validity** -- Information has a shelf life. A contract from 2022 may have been superseded by a 2024 amendment.

Let's build systems that address all three.

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_03_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 2: Building Intuition -- The Provenance Problem

In [ ]:
# ============================================================
# 2A: Claims without provenance -- the danger
# ============================================================

# Imagine an agent extracted these claims from a batch of contracts.
# Without provenance, they are all equally unverifiable.

claims_without_provenance = [
    "Payment deadline is March 31, 2024",
    "Liability cap is $5,000,000",
    "Contract auto-renews annually",
    "Termination requires 90 days written notice",
    "Governing law is State of Delaware",
    "Insurance minimum is $2,000,000 per occurrence",
    "Non-compete period is 24 months",   # <-- Actually hallucinated!
    "Arbitration clause applies to all disputes",
]

print("EXTRACTED CLAIMS (no provenance):")
print("=" * 60)
for i, claim in enumerate(claims_without_provenance, 1):
    print(f"  {i}. {claim}")

print("\n\nPROBLEM: Which of these came from an actual document?")
print("Claim #7 was hallucinated -- the contract says 12 months,")
print("not 24. But without provenance, you cannot tell.")
print("\nA human reviewer would need to re-read all 50 contracts")
print("to verify each claim. That defeats the purpose of automation.")

In [ ]:
# ============================================================
# 2B: The same claims WITH provenance
# ============================================================

claims_with_provenance = [
    {
        "claim": "Payment deadline is March 31, 2024",
        "value": "2024-03-31",
        "field_type": "deadline",
        "source": {
            "document_id": "contract_2024_047",
            "document_title": "Service Agreement - Acme Corp",
            "page": 12,
            "paragraph": 3,
            "exact_quote": "All payments shall be remitted no later "
                          "than March 31, 2024.",
            "extraction_timestamp": "2024-03-15T14:22:00Z",
        },
        "confidence": 0.97,
    },
    {
        "claim": "Liability cap is $5,000,000",
        "value": 5000000,
        "field_type": "liability_cap",
        "source": {
            "document_id": "contract_2024_047",
            "document_title": "Service Agreement - Acme Corp",
            "page": 8,
            "paragraph": 1,
            "exact_quote": "The total aggregate liability of either "
                          "party shall not exceed Five Million Dollars "
                          "($5,000,000).",
            "extraction_timestamp": "2024-03-15T14:22:05Z",
        },
        "confidence": 0.99,
    },
    {
        "claim": "Non-compete period is 24 months",
        "value": 24,
        "field_type": "non_compete_months",
        "source": {
            "document_id": "NONE",
            "document_title": "NONE",
            "page": None,
            "paragraph": None,
            "exact_quote": None,  # <-- No source quote!
            "extraction_timestamp": "2024-03-15T14:22:30Z",
        },
        "confidence": 0.31,  # <-- Very low confidence
    },
]

print("CLAIMS WITH PROVENANCE:")
print("=" * 60)
for claim_data in claims_with_provenance:
    src = claim_data["source"]
    has_source = src["exact_quote"] is not None
    status = "VERIFIED" if has_source else "FLAGGED -- NO SOURCE"

    print(f"\n  Claim: \"{claim_data['claim']}\"")
    print(f"  Status: {status}")
    print(f"  Confidence: {claim_data['confidence']:.2f}")
    if has_source:
        print(f"  Source: {src['document_title']}, "
              f"page {src['page']}, paragraph {src['paragraph']}")
        print(f"  Quote: \"{src['exact_quote'][:60]}...\"")
    else:
        print(f"  Source: MISSING -- this claim cannot be verified!")

print("\n\nClaim #3 is immediately flagged: no source document, no quote,")
print("confidence of 0.31. A reviewer knows to check this one.")
print("Without provenance, it would have been treated as fact.")

### The Core Principle

**Every extracted claim must be linked to a specific document, page, and paragraph.** If the model cannot cite its source, the claim is flagged for human review. This transforms hallucination from a **silent failure** into a **detectable event.**

---

In [ ]:
#@title 🎧 Listen: Claim Source Mapping
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_04_claim_source_mapping.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Temporal Validity
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_05_temporal_validity.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Conflict Detection
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_06_conflict_detection.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 3: Core Implementation -- Claim-Source Mapping System

In [ ]:
# ============================================================
# 3A: Define the data models
# ============================================================

@dataclass
class SourceReference:
    """Where a claim came from."""
    document_id: str
    document_title: str
    page: Optional[int] = None
    paragraph: Optional[int] = None
    section: Optional[str] = None
    exact_quote: Optional[str] = None
    document_date: Optional[datetime] = None
    extraction_timestamp: Optional[datetime] = None

    def is_valid(self) -> bool:
        """A source is valid if it has at minimum a document and a quote."""
        return (
            bool(self.document_id) and
            self.document_id != "NONE" and
            bool(self.exact_quote)
        )


@dataclass
class ExtractedClaim:
    """A single claim extracted from a source document."""
    claim_id: str
    claim_text: str
    value: Any
    field_type: str
    source: SourceReference
    confidence: float
    temporal_warnings: list = field(default_factory=list)
    verification_status: str = "pending"  # pending, verified, flagged, rejected

    def flag_if_unsourced(self) -> None:
        """Flag claims that have no valid source."""
        if not self.source.is_valid():
            self.verification_status = "flagged"
            self.temporal_warnings.append({
                "type": "missing_source",
                "message": "No valid source reference. Possible hallucination.",
                "severity": "HIGH",
            })


@dataclass
class ExtractionBatch:
    """A batch of claims extracted from multiple documents."""
    batch_id: str
    created_at: datetime
    claims: list = field(default_factory=list)

    def add_claim(self, claim: ExtractedClaim) -> None:
        """Add a claim to the batch, automatically flagging unsourced ones."""
        claim.flag_if_unsourced()
        self.claims.append(claim)

    def get_flagged(self) -> list:
        """Return all flagged claims."""
        return [c for c in self.claims if c.verification_status == "flagged"]

    def get_verified(self) -> list:
        """Return all verified claims."""
        return [c for c in self.claims if c.verification_status == "verified"]

    def summary(self) -> dict:
        """Summary statistics for this extraction batch."""
        status_counts = defaultdict(int)
        for c in self.claims:
            status_counts[c.verification_status] += 1
        avg_confidence = (
            sum(c.confidence for c in self.claims) / len(self.claims)
            if self.claims else 0
        )
        return {
            "batch_id": self.batch_id,
            "total_claims": len(self.claims),
            "status_breakdown": dict(status_counts),
            "average_confidence": round(avg_confidence, 3),
            "flagged_count": len(self.get_flagged()),
        }


# Create a demonstration batch
batch = ExtractionBatch(batch_id="batch_2024_03_15", created_at=NOW)

# Add well-sourced claims
claim_1 = ExtractedClaim(
    claim_id="CLM-001",
    claim_text="Payment deadline is March 31, 2024",
    value="2024-03-31",
    field_type="deadline",
    source=SourceReference(
        document_id="contract_2024_047",
        document_title="Service Agreement - Acme Corp",
        page=12, paragraph=3,
        exact_quote="All payments shall be remitted no later than March 31, 2024.",
        document_date=datetime(2024, 1, 15),
        extraction_timestamp=NOW,
    ),
    confidence=0.97,
)

claim_2 = ExtractedClaim(
    claim_id="CLM-002",
    claim_text="Liability cap is $5,000,000",
    value=5000000,
    field_type="liability_cap",
    source=SourceReference(
        document_id="contract_2024_047",
        document_title="Service Agreement - Acme Corp",
        page=8, paragraph=1,
        exact_quote="Total aggregate liability shall not exceed $5,000,000.",
        document_date=datetime(2024, 1, 15),
        extraction_timestamp=NOW,
    ),
    confidence=0.99,
)

# Add an unsourced claim (will be flagged automatically)
claim_3 = ExtractedClaim(
    claim_id="CLM-003",
    claim_text="Non-compete period is 24 months",
    value=24,
    field_type="non_compete_months",
    source=SourceReference(
        document_id="NONE",
        document_title="NONE",
        extraction_timestamp=NOW,
    ),
    confidence=0.31,
)

batch.add_claim(claim_1)
batch.add_claim(claim_2)
batch.add_claim(claim_3)

print("EXTRACTION BATCH SUMMARY:")
print("=" * 60)
pp(batch.summary())

print("\nFLAGGED CLAIMS:")
for c in batch.get_flagged():
    print(f"  [{c.claim_id}] \"{c.claim_text}\"")
    print(f"    Confidence: {c.confidence}")
    print(f"    Warnings: {c.temporal_warnings}")

In [ ]:
# ============================================================
# 3B: Build the temporal validity checker
# ============================================================

def check_temporal_validity(
    claim: ExtractedClaim,
    current_time: datetime = NOW,
    max_document_age_days: int = 365,
    max_extraction_age_days: int = 90,
) -> list[dict]:
    """
    Check whether a claim's source is still temporally valid.

    Flags:
    1. Source document older than max_document_age_days
    2. Extraction older than max_extraction_age_days
    3. Source document has no date (unknown freshness)

    Returns a list of warning dicts.
    """
    warnings = []

    # Check 1: Document age
    doc_date = claim.source.document_date
    if doc_date:
        age_days = (current_time - doc_date).days
        if age_days > max_document_age_days:
            warnings.append({
                "type": "outdated_source",
                "severity": "HIGH",
                "message": (
                    f"Source document is {age_days} days old "
                    f"(threshold: {max_document_age_days}). "
                    f"Document date: {doc_date.strftime('%Y-%m-%d')}."
                ),
                "recommendation": "Verify against the current version of this document.",
            })
        elif age_days > max_document_age_days * 0.75:
            warnings.append({
                "type": "aging_source",
                "severity": "MEDIUM",
                "message": (
                    f"Source document is {age_days} days old "
                    f"(approaching {max_document_age_days}-day threshold)."
                ),
                "recommendation": "Schedule re-extraction before threshold.",
            })
    else:
        warnings.append({
            "type": "unknown_document_date",
            "severity": "MEDIUM",
            "message": "Source document has no date. Cannot verify freshness.",
            "recommendation": "Obtain document date and re-evaluate.",
        })

    # Check 2: Extraction age
    ext_date = claim.source.extraction_timestamp
    if ext_date:
        ext_age_days = (current_time - ext_date).days
        if ext_age_days > max_extraction_age_days:
            warnings.append({
                "type": "stale_extraction",
                "severity": "MEDIUM",
                "message": (
                    f"Extraction is {ext_age_days} days old "
                    f"(threshold: {max_extraction_age_days})."
                ),
                "recommendation": "Re-extract from the latest version.",
            })

    return warnings


# Test with claims of different ages
test_claims = [
    ExtractedClaim(
        claim_id="FRESH-001",
        claim_text="Coverage limit is $100,000",
        value=100000, field_type="coverage_limit",
        source=SourceReference(
            document_id="doc_fresh",
            document_title="Recent Policy",
            page=5, paragraph=2,
            exact_quote="Coverage shall not exceed $100,000.",
            document_date=datetime(2024, 2, 1),
            extraction_timestamp=datetime(2024, 3, 10),
        ),
        confidence=0.95,
    ),
    ExtractedClaim(
        claim_id="OLD-002",
        claim_text="Deductible is $500",
        value=500, field_type="deductible",
        source=SourceReference(
            document_id="doc_old",
            document_title="Legacy Policy 2022",
            page=3, paragraph=1,
            exact_quote="The deductible amount is Five Hundred Dollars ($500).",
            document_date=datetime(2022, 6, 15),
            extraction_timestamp=datetime(2023, 8, 1),
        ),
        confidence=0.92,
    ),
    ExtractedClaim(
        claim_id="NODATE-003",
        claim_text="Arbitration is mandatory",
        value=True, field_type="arbitration_required",
        source=SourceReference(
            document_id="doc_nodate",
            document_title="Unknown Date Contract",
            page=14, paragraph=7,
            exact_quote="All disputes shall be resolved by binding arbitration.",
        ),
        confidence=0.88,
    ),
]

print("TEMPORAL VALIDITY CHECK:")
print("=" * 65)
for claim in test_claims:
    warnings = check_temporal_validity(claim)
    status = "FRESH" if not warnings else "WARNING"
    print(f"\n  [{claim.claim_id}] \"{claim.claim_text}\"")
    print(f"  Status: {status}")
    if warnings:
        for w in warnings:
            print(f"    [{w['severity']}] {w['type']}: {w['message']}")
            print(f"         Rec: {w['recommendation']}")
    else:
        print(f"    All temporal checks passed.")

In [ ]:
# ============================================================
# 3C: Conflict detection -- when two sources disagree
# ============================================================

def detect_conflicts(claims: list[ExtractedClaim]) -> list[dict]:
    """
    Find claims about the same field that have conflicting values.

    Groups claims by field_type and checks for value disagreements.
    """
    # Group by field type
    by_field = defaultdict(list)
    for claim in claims:
        by_field[claim.field_type].append(claim)

    conflicts = []
    for field_type, field_claims in by_field.items():
        if len(field_claims) < 2:
            continue

        # Check all pairs for value conflicts
        for i in range(len(field_claims)):
            for j in range(i + 1, len(field_claims)):
                c1, c2 = field_claims[i], field_claims[j]
                if c1.value != c2.value:
                    # Determine which is more trustworthy
                    if c1.confidence > c2.confidence:
                        preferred, alternate = c1, c2
                    else:
                        preferred, alternate = c2, c1

                    conflicts.append({
                        "field_type": field_type,
                        "claim_1": {
                            "id": c1.claim_id,
                            "value": c1.value,
                            "source": c1.source.document_title,
                            "confidence": c1.confidence,
                        },
                        "claim_2": {
                            "id": c2.claim_id,
                            "value": c2.value,
                            "source": c2.source.document_title,
                            "confidence": c2.confidence,
                        },
                        "preferred": preferred.claim_id,
                        "reason": (
                            f"Higher confidence ({preferred.confidence:.2f} "
                            f"vs {alternate.confidence:.2f})"
                        ),
                    })

    return conflicts


# Demonstrate with conflicting claims
conflicting_claims = [
    ExtractedClaim(
        claim_id="A-001",
        claim_text="Termination notice is 30 days",
        value=30, field_type="termination_notice_days",
        source=SourceReference(
            document_id="contract_v1",
            document_title="Original Contract (Jan 2023)",
            page=15, paragraph=4,
            exact_quote="Either party may terminate with 30 days written notice.",
            document_date=datetime(2023, 1, 10),
            extraction_timestamp=NOW,
        ),
        confidence=0.94,
    ),
    ExtractedClaim(
        claim_id="A-002",
        claim_text="Termination notice is 90 days",
        value=90, field_type="termination_notice_days",
        source=SourceReference(
            document_id="amendment_v2",
            document_title="Amendment #1 (Sep 2023)",
            page=2, paragraph=1,
            exact_quote="Section 12.1 is hereby amended: termination "
                       "requires 90 days written notice.",
            document_date=datetime(2023, 9, 1),
            extraction_timestamp=NOW,
        ),
        confidence=0.96,
    ),
]

conflicts = detect_conflicts(conflicting_claims)
print("CONFLICT DETECTION:")
print("=" * 65)
for conflict in conflicts:
    print(f"\n  Field: {conflict['field_type']}")
    print(f"  Claim 1: {conflict['claim_1']['value']} days "
          f"(from \"{conflict['claim_1']['source']}\", "
          f"conf={conflict['claim_1']['confidence']})")
    print(f"  Claim 2: {conflict['claim_2']['value']} days "
          f"(from \"{conflict['claim_2']['source']}\", "
          f"conf={conflict['claim_2']['confidence']})")
    print(f"  Preferred: {conflict['preferred']} -- {conflict['reason']}")
    print(f"\n  Note: In this case, the amendment supersedes the original.")
    print(f"  The system correctly prefers Claim A-002 (higher confidence),")
    print(f"  AND a human reviewer can verify by checking both source documents.")

In [ ]:
#@title 🎧 Listen: Provenance Tracker Todo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_07_provenance_tracker_todo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Verification Todo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_08_verification_todo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 4: Exercise -- Build a Complete Provenance Tracker

In [ ]:
# ============================================================
# 4A: TODO -- Implement the ProvenanceTracker class
# ============================================================

# TODO: Complete the ProvenanceTracker class below.
# It should:
# 1. Add claims with automatic source validation
# 2. Run temporal checks on all claims
# 3. Detect conflicts between claims
# 4. Generate an audit report

class ProvenanceTracker:
    """
    Tracks claim-source mappings, temporal validity, and conflicts
    across an extraction batch.
    """

    def __init__(self, batch_id: str, current_time: datetime = NOW):
        self.batch_id = batch_id
        self.current_time = current_time
        self.claims: list[ExtractedClaim] = []
        self.audit_log: list[dict] = []

    def add_claim(self, claim: ExtractedClaim) -> str:
        """
        Add a claim to the tracker.

        TODO 1: Flag the claim if it has no valid source.
        TODO 2: Run temporal validity checks.
        TODO 3: Log the addition to the audit_log.

        Returns the verification_status of the claim.
        """
        # TODO 1: Call claim.flag_if_unsourced()

        # TODO 2: Run check_temporal_validity and add warnings to claim

        # TODO 3: Add to audit log with timestamp and action

        self.claims.append(claim)
        return claim.verification_status

    def find_conflicts(self) -> list[dict]:
        """
        TODO 4: Find all conflicts between claims in this tracker.
        Use the detect_conflicts function from Section 3C.
        """
        return []  # <-- Replace with actual implementation

    def audit_report(self) -> dict:
        """
        TODO 5: Generate an audit report summarizing:
        - Total claims
        - Verified / flagged / pending counts
        - Average confidence
        - Number of conflicts
        - Number of temporal warnings
        """
        return {}  # <-- Replace with actual implementation


# Test your implementation
tracker = ProvenanceTracker("test_batch_001")

# Add some claims
tracker.add_claim(claim_1)
tracker.add_claim(claim_2)
tracker.add_claim(claim_3)

# Add the conflicting claims
for c in conflicting_claims:
    tracker.add_claim(c)

print("YOUR AUDIT REPORT:")
pp(tracker.audit_report())

print("\nCONFLICTS FOUND:")
pp(tracker.find_conflicts())

In [ ]:
# ============================================================
# 4B: SOLUTION -- ProvenanceTracker
# ============================================================

class ProvenanceTrackerSolution:
    """
    Tracks claim-source mappings, temporal validity, and conflicts.
    """

    def __init__(self, batch_id: str, current_time: datetime = NOW):
        self.batch_id = batch_id
        self.current_time = current_time
        self.claims: list[ExtractedClaim] = []
        self.audit_log: list[dict] = []

    def add_claim(self, claim: ExtractedClaim) -> str:
        """Add a claim with automatic validation and temporal checks."""
        # 1. Flag if unsourced
        claim.flag_if_unsourced()

        # 2. Run temporal validity checks
        warnings = check_temporal_validity(claim, self.current_time)
        claim.temporal_warnings.extend(warnings)

        # If there are HIGH severity temporal warnings, flag the claim
        high_warnings = [w for w in warnings if w.get("severity") == "HIGH"]
        if high_warnings and claim.verification_status != "flagged":
            claim.verification_status = "flagged"

        # If source is valid and no high warnings, mark as verified
        if (claim.source.is_valid() and
            not high_warnings and
            claim.verification_status == "pending"):
            claim.verification_status = "verified"

        # 3. Audit log entry
        self.audit_log.append({
            "timestamp": self.current_time.isoformat(),
            "action": "claim_added",
            "claim_id": claim.claim_id,
            "status": claim.verification_status,
            "confidence": claim.confidence,
            "source_valid": claim.source.is_valid(),
            "temporal_warnings_count": len(warnings),
        })

        self.claims.append(claim)
        return claim.verification_status

    def find_conflicts(self) -> list[dict]:
        """Find all value conflicts between claims."""
        return detect_conflicts(self.claims)

    def audit_report(self) -> dict:
        """Generate a comprehensive audit report."""
        status_counts = defaultdict(int)
        total_warnings = 0
        for c in self.claims:
            status_counts[c.verification_status] += 1
            total_warnings += len(c.temporal_warnings)

        avg_confidence = (
            sum(c.confidence for c in self.claims) / len(self.claims)
            if self.claims else 0
        )

        conflicts = self.find_conflicts()

        return {
            "batch_id": self.batch_id,
            "report_generated": self.current_time.isoformat(),
            "total_claims": len(self.claims),
            "status_breakdown": dict(status_counts),
            "average_confidence": round(avg_confidence, 3),
            "total_temporal_warnings": total_warnings,
            "conflicts_found": len(conflicts),
            "conflict_details": conflicts,
            "audit_log_entries": len(self.audit_log),
        }


# Run the solution
tracker_sol = ProvenanceTrackerSolution("test_batch_001")

# We need fresh copies of claims since dataclass instances are mutable
fresh_claims = [
    ExtractedClaim(
        claim_id="CLM-001",
        claim_text="Payment deadline is March 31, 2024",
        value="2024-03-31", field_type="deadline",
        source=SourceReference(
            document_id="contract_2024_047",
            document_title="Service Agreement - Acme Corp",
            page=12, paragraph=3,
            exact_quote="All payments shall be remitted no later than March 31, 2024.",
            document_date=datetime(2024, 1, 15),
            extraction_timestamp=NOW,
        ),
        confidence=0.97,
    ),
    ExtractedClaim(
        claim_id="CLM-002",
        claim_text="Liability cap is $5,000,000",
        value=5000000, field_type="liability_cap",
        source=SourceReference(
            document_id="contract_2024_047",
            document_title="Service Agreement - Acme Corp",
            page=8, paragraph=1,
            exact_quote="Total aggregate liability shall not exceed $5,000,000.",
            document_date=datetime(2024, 1, 15),
            extraction_timestamp=NOW,
        ),
        confidence=0.99,
    ),
    ExtractedClaim(
        claim_id="CLM-003",
        claim_text="Non-compete period is 24 months",
        value=24, field_type="non_compete_months",
        source=SourceReference(
            document_id="NONE",
            document_title="NONE",
            extraction_timestamp=NOW,
        ),
        confidence=0.31,
    ),
    ExtractedClaim(
        claim_id="CLM-004",
        claim_text="Termination notice is 30 days",
        value=30, field_type="termination_notice_days",
        source=SourceReference(
            document_id="contract_v1",
            document_title="Original Contract (Jan 2023)",
            page=15, paragraph=4,
            exact_quote="Either party may terminate with 30 days written notice.",
            document_date=datetime(2023, 1, 10),
            extraction_timestamp=NOW,
        ),
        confidence=0.94,
    ),
    ExtractedClaim(
        claim_id="CLM-005",
        claim_text="Termination notice is 90 days",
        value=90, field_type="termination_notice_days",
        source=SourceReference(
            document_id="amendment_v2",
            document_title="Amendment #1 (Sep 2023)",
            page=2, paragraph=1,
            exact_quote="Section 12.1 is hereby amended: termination "
                       "requires 90 days written notice.",
            document_date=datetime(2023, 9, 1),
            extraction_timestamp=NOW,
        ),
        confidence=0.96,
    ),
]

for c in fresh_claims:
    status = tracker_sol.add_claim(c)
    print(f"  Added {c.claim_id}: {status}")

print("\nAUDIT REPORT:")
pp(tracker_sol.audit_report())

In [ ]:
# ============================================================
# 4C: TODO -- Claim verification workflow
# ============================================================

# TODO: Implement a function that takes a claim and a "ground truth"
# document lookup, and verifies whether the claim's exact_quote
# actually appears in the cited document.

def verify_claim_against_source(
    claim: ExtractedClaim,
    document_store: dict,
) -> dict:
    """
    Verify that a claim's exact_quote actually appears in the
    cited document.

    Args:
        claim: The claim to verify
        document_store: dict mapping document_id -> document text

    Returns:
        Verification result dict with status and details
    """
    # TODO 1: Check if the source document exists in the store

    # TODO 2: Check if the exact_quote appears in the document text

    # TODO 3: Return a structured verification result:
    #   - "verified" if quote found in document
    #   - "quote_not_found" if document exists but quote is missing
    #   - "document_not_found" if document_id is not in the store

    return {
        "claim_id": claim.claim_id,
        "status": "not_implemented",
    }


# Test data: mock document store
mock_documents = {
    "contract_2024_047": (
        "Page 1: This Service Agreement is between...\n"
        "Page 8: The total aggregate liability of either party "
        "shall not exceed Five Million Dollars ($5,000,000).\n"
        "Page 12: All payments shall be remitted no later "
        "than March 31, 2024.\n"
        "Page 15: This agreement shall be governed by Delaware law."
    ),
    "contract_v1": (
        "Page 15: Either party may terminate with 30 days written notice."
    ),
    "amendment_v2": (
        "Page 2: Section 12.1 is hereby amended: termination "
        "requires 90 days written notice."
    ),
}

print("YOUR VERIFICATION RESULTS:")
for c in fresh_claims:
    result = verify_claim_against_source(c, mock_documents)
    print(f"  {result['claim_id']}: {result['status']}")

In [ ]:
# ============================================================
# 4D: SOLUTION -- Claim verification workflow
# ============================================================

def verify_claim_against_source_solution(
    claim: ExtractedClaim,
    document_store: dict,
) -> dict:
    """Verify that a claim's exact_quote appears in the cited document."""
    doc_id = claim.source.document_id
    quote = claim.source.exact_quote

    # Check 1: Document exists?
    if not doc_id or doc_id == "NONE" or doc_id not in document_store:
        return {
            "claim_id": claim.claim_id,
            "status": "document_not_found",
            "message": f"Document '{doc_id}' not in document store.",
            "recommendation": "Cannot verify. Flag for manual review.",
        }

    doc_text = document_store[doc_id]

    # Check 2: No quote to verify?
    if not quote:
        return {
            "claim_id": claim.claim_id,
            "status": "no_quote_provided",
            "message": "Claim has no exact_quote to verify.",
            "recommendation": "Re-extract with source quoting enabled.",
        }

    # Check 3: Quote appears in document?
    # Use case-insensitive partial match (quotes may have minor formatting diffs)
    if quote.lower() in doc_text.lower():
        return {
            "claim_id": claim.claim_id,
            "status": "verified",
            "message": f"Quote found in '{doc_id}'.",
            "matched_text": quote[:80] + "..." if len(quote) > 80 else quote,
        }
    else:
        # Try a fuzzy approach: check if 80% of words match
        quote_words = set(quote.lower().split())
        doc_words = set(doc_text.lower().split())
        overlap = len(quote_words & doc_words) / len(quote_words) if quote_words else 0

        if overlap > 0.8:
            return {
                "claim_id": claim.claim_id,
                "status": "partial_match",
                "message": (
                    f"Quote partially matches ({overlap:.0%} word overlap). "
                    f"Possible formatting difference."
                ),
                "recommendation": "Manual review recommended.",
            }
        else:
            return {
                "claim_id": claim.claim_id,
                "status": "quote_not_found",
                "message": (
                    f"Quote not found in '{doc_id}' "
                    f"(word overlap: {overlap:.0%})."
                ),
                "recommendation": "Possible hallucination. Flag for review.",
            }


print("SOLUTION -- VERIFICATION RESULTS:")
print("=" * 65)
for c in fresh_claims:
    result = verify_claim_against_source_solution(c, mock_documents)
    status_icon = {
        "verified": "[OK]",
        "partial_match": "[~~]",
        "quote_not_found": "[!!]",
        "document_not_found": "[??]",
        "no_quote_provided": "[--]",
    }.get(result["status"], "[??]")
    print(f"\n  {status_icon} {result['claim_id']}: {result['status']}")
    print(f"      {result['message']}")
    if "recommendation" in result:
        print(f"      Rec: {result['recommendation']}")

In [ ]:
#@title 🎧 Listen: Scratchpads
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_09_scratchpads.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Crash Recovery Demo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_10_crash_recovery_demo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 5: Scratchpad Files -- Preserving State Across Turns

When an agent processes 50 documents one by one, each intermediate result adds to the context. By document 30, the context window is full and earlier results start getting dropped. **Scratchpad files** solve this by offloading intermediate results to persistent storage.

In [ ]:
# ============================================================
# 5A: The scratchpad schema
# ============================================================

def create_scratchpad(task_id: str, total_documents: int) -> dict:
    """
    Create a new scratchpad for a multi-document analysis task.

    The scratchpad stores:
    - Task metadata (ID, timestamps, configuration)
    - Processing progress (which documents are done)
    - Accumulated results (extracted claims so far)
    - Running statistics (totals, averages, flags)
    - Notes (observations that do not fit neatly into structured fields)
    """
    return {
        "task_id": task_id,
        "created_at": NOW.isoformat(),
        "last_updated": NOW.isoformat(),
        "current_step": "document_analysis",
        "document_index": 0,
        "total_documents": total_documents,
        "config": {
            "extraction_model": "claude-sonnet-4-5-20250514",
            "confidence_threshold": 0.7,
            "max_retries_per_document": 3,
        },
        "accumulated_results": {
            "processed": 0,
            "findings": [],
        },
        "running_statistics": {
            "total_claims_extracted": 0,
            "total_claims_value": 0.0,
            "flagged_for_review": 0,
            "conflicts_detected": 0,
            "average_confidence": 0.0,
            "documents_with_errors": 0,
        },
        "notes": [],
        "error_log": [],
    }


# Create a scratchpad
scratchpad = create_scratchpad("insurance-batch-2024-03-15", 50)

print("SCRATCHPAD SCHEMA:")
print("=" * 60)
pp(scratchpad)

print("\nKey design choices:")
print("  1. document_index tracks where we are (for crash recovery)")
print("  2. accumulated_results stores per-document findings")
print("  3. running_statistics keeps aggregates without re-reading all results")
print("  4. notes captures freeform observations")
print("  5. error_log tracks failures for retry")

In [ ]:
# ============================================================
# 5B: The read-write-checkpoint pattern
# ============================================================

def simulate_document_extraction(doc_id: str, doc_index: int) -> dict:
    """
    Simulate extracting claims from a single document.
    Returns a findings dict with claims, confidence, and metadata.
    """
    # Simulate different outcomes
    random.seed(hash(doc_id) % 10000)

    num_claims = random.randint(1, 5)
    claims = []
    risk_flags = []

    for i in range(num_claims):
        confidence = random.uniform(0.4, 0.99)
        claim_value = random.uniform(100, 500000)
        claims.append({
            "claim_text": f"Claim from {doc_id}, item {i+1}",
            "value": round(claim_value, 2),
            "confidence": round(confidence, 3),
            "page": random.randint(1, 30),
            "has_source_quote": confidence > 0.5,
        })
        if confidence < 0.7:
            risk_flags.append(
                f"Low confidence ({confidence:.2f}) on claim item {i+1}"
            )

    # Simulate occasional errors
    error = None
    if random.random() < 0.05:
        error = {
            "type": "extraction_timeout",
            "message": f"Extraction timed out for {doc_id}",
            "retry_recommended": True,
        }

    return {
        "doc_id": doc_id,
        "doc_index": doc_index,
        "claims": claims,
        "risk_flags": risk_flags,
        "error": error,
    }


def update_scratchpad(scratchpad: dict, result: dict) -> dict:
    """
    Update the scratchpad with results from one document.
    This is the write step of the read-write-checkpoint pattern.
    """
    sp = scratchpad  # alias for brevity

    # Update progress
    sp["document_index"] = result["doc_index"] + 1
    sp["accumulated_results"]["processed"] += 1
    sp["last_updated"] = NOW.isoformat()

    if result["error"]:
        # Log the error but continue
        sp["error_log"].append(result["error"])
        sp["running_statistics"]["documents_with_errors"] += 1
        return sp

    # Add findings
    sp["accumulated_results"]["findings"].append({
        "doc_id": result["doc_id"],
        "num_claims": len(result["claims"]),
        "risk_flags": result["risk_flags"],
        "avg_confidence": round(
            sum(c["confidence"] for c in result["claims"]) /
            len(result["claims"]) if result["claims"] else 0, 3
        ),
    })

    # Update running statistics
    stats = sp["running_statistics"]
    stats["total_claims_extracted"] += len(result["claims"])
    total_value = sum(c["value"] for c in result["claims"])
    stats["total_claims_value"] += total_value
    stats["flagged_for_review"] += len(result["risk_flags"])

    # Update running average confidence
    all_count = stats["total_claims_extracted"]
    if all_count > 0:
        # Weighted update: blend old avg with new claims
        old_count = all_count - len(result["claims"])
        old_avg = stats["average_confidence"]
        new_avg = sum(c["confidence"] for c in result["claims"]) / len(result["claims"])
        stats["average_confidence"] = round(
            (old_avg * old_count + new_avg * len(result["claims"])) / all_count,
            3
        )

    # Add notes for unusual findings
    if result["risk_flags"]:
        sp["notes"].append(
            f"Doc {result['doc_id']}: {len(result['risk_flags'])} risk flags"
        )

    return sp


# Process 10 documents to demonstrate the pattern
scratchpad = create_scratchpad("demo-batch", 50)

print("PROCESSING DOCUMENTS (first 10 of 50):")
print("=" * 65)

for i in range(10):
    doc_id = f"doc_{i+1:03d}"
    result = simulate_document_extraction(doc_id, i)

    status = "ERROR" if result["error"] else "OK"
    claim_count = len(result["claims"])
    flags = len(result["risk_flags"])

    print(f"  [{i+1:2d}/50] {doc_id}: {status} | "
          f"{claim_count} claims | {flags} flags")

    scratchpad = update_scratchpad(scratchpad, result)

print(f"\nSCRATCHPAD STATE AFTER 10 DOCUMENTS:")
print(f"  Processed: {scratchpad['accumulated_results']['processed']}/50")
print(f"  Total claims: {scratchpad['running_statistics']['total_claims_extracted']}")
print(f"  Avg confidence: {scratchpad['running_statistics']['average_confidence']:.3f}")
print(f"  Flagged: {scratchpad['running_statistics']['flagged_for_review']}")
print(f"  Errors: {scratchpad['running_statistics']['documents_with_errors']}")
print(f"  Notes: {len(scratchpad['notes'])}")

In [ ]:
# ============================================================
# 5C: Crash recovery demonstration
# ============================================================

# Simulate a crash at document 10, then resume
print("CRASH RECOVERY DEMONSTRATION:")
print("=" * 65)

# The scratchpad was saved after document 10
# Pretend the system crashed and restarted
print(f"\n  [CRASH] System restarted.")
print(f"  Reading scratchpad from disk...")
print(f"  Last checkpoint: document_index = {scratchpad['document_index']}")
print(f"  Resuming from document {scratchpad['document_index'] + 1}...")

# Resume from where we left off
start_index = scratchpad["document_index"]
for i in range(start_index, min(start_index + 5, 50)):
    doc_id = f"doc_{i+1:03d}"
    result = simulate_document_extraction(doc_id, i)
    status = "ERROR" if result["error"] else "OK"
    print(f"  [{i+1:2d}/50] {doc_id}: {status} | "
          f"{len(result['claims'])} claims")
    scratchpad = update_scratchpad(scratchpad, result)

print(f"\nSCRATCHPAD STATE AFTER RECOVERY:")
print(f"  Processed: {scratchpad['accumulated_results']['processed']}/50")
print(f"  Total claims: {scratchpad['running_statistics']['total_claims_extracted']}")
print(f"  No work was lost. The 10 documents processed before the crash")
print(f"  are still in the scratchpad. We only processed the remaining ones.")

In [ ]:
#@title 🎧 Listen: Scratchpad Design Todo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_11_scratchpad_design_todo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 6: Exercise -- Scratchpad Design for a Real Scenario

In [ ]:
# ============================================================
# 6A: TODO -- Design a scratchpad for multi-step legal analysis
# ============================================================

# Scenario: You are building an agent that reviews 100 legal contracts
# to produce a compliance report. The process has 3 stages:
#
# Stage 1: Extract key terms from each contract
# Stage 2: Cross-reference terms to find conflicts between contracts
# Stage 3: Generate a compliance summary with risk scores
#
# TODO: Design a scratchpad schema that supports all 3 stages,
# including crash recovery at any stage.

def create_legal_analysis_scratchpad(
    task_id: str,
    contracts: list[str],
) -> dict:
    """
    Design a scratchpad for multi-stage legal analysis.

    TODO 1: Add a "stages" field that tracks which stage we are on
    and the progress within each stage.

    TODO 2: Add a "stage_1_results" field for extracted terms.

    TODO 3: Add a "stage_2_results" field for cross-reference findings.

    TODO 4: Add a "stage_3_results" field for the compliance summary.

    TODO 5: Add a "checkpoint" field that records the exact resume
    point (stage + index within that stage).
    """
    return {
        "task_id": task_id,
        "total_contracts": len(contracts),
        "contracts": contracts,
        "created_at": NOW.isoformat(),
        # TODO: Add your fields here
    }


# Test
contracts = [f"contract_{i:03d}" for i in range(1, 101)]
sp = create_legal_analysis_scratchpad("legal_review_001", contracts)
print("YOUR SCRATCHPAD:")
pp(sp)

In [ ]:
# ============================================================
# 6B: SOLUTION -- Legal analysis scratchpad
# ============================================================

def create_legal_analysis_scratchpad_solution(
    task_id: str,
    contracts: list[str],
) -> dict:
    """Multi-stage scratchpad for legal contract analysis."""
    return {
        "task_id": task_id,
        "total_contracts": len(contracts),
        "contracts": contracts,
        "created_at": NOW.isoformat(),
        "last_updated": NOW.isoformat(),

        # Stage tracking
        "current_stage": 1,
        "stages": {
            "stage_1_extraction": {
                "status": "in_progress",
                "progress": 0,
                "total": len(contracts),
                "description": "Extract key terms from each contract",
            },
            "stage_2_cross_reference": {
                "status": "pending",
                "progress": 0,
                "total": 0,  # Set after stage 1 completes
                "description": "Cross-reference terms to find conflicts",
            },
            "stage_3_compliance": {
                "status": "pending",
                "progress": 0,
                "total": 1,  # Single output
                "description": "Generate compliance summary with risk scores",
            },
        },

        # Stage 1 results: per-contract extracted terms
        "stage_1_results": {
            "extracted_terms": [],
            # Each entry: {contract_id, terms: [...], confidence, warnings}
        },

        # Stage 2 results: cross-reference findings
        "stage_2_results": {
            "conflicts": [],
            # Each: {contract_a, contract_b, field, value_a, value_b, severity}
            "consistent_terms": [],
            # Terms that agree across all contracts
        },

        # Stage 3 results: compliance summary
        "stage_3_results": {
            "overall_risk_score": None,
            "per_contract_risk": [],
            "critical_issues": [],
            "recommendations": [],
        },

        # Crash recovery checkpoint
        "checkpoint": {
            "stage": 1,
            "index": 0,  # Within-stage progress
            "last_successful_item": None,
            "timestamp": NOW.isoformat(),
        },

        # Running statistics
        "statistics": {
            "total_terms_extracted": 0,
            "total_conflicts_found": 0,
            "average_confidence": 0.0,
            "high_risk_contracts": 0,
        },

        "error_log": [],
        "notes": [],
    }


sp_solution = create_legal_analysis_scratchpad_solution(
    "legal_review_001", contracts
)

print("SOLUTION -- LEGAL ANALYSIS SCRATCHPAD:")
print("=" * 65)
# Print schema without the full contracts list (too long)
sp_display = {k: v for k, v in sp_solution.items() if k != "contracts"}
sp_display["contracts"] = f"[{len(contracts)} contracts]"
pp(sp_display)

print("\nKey design choices:")
print("  1. Each stage has its own status, progress, and results section")
print("  2. The checkpoint records the exact resume point (stage + index)")
print("  3. Running statistics are updated incrementally (no re-computation)")
print("  4. Error log and notes capture issues without blocking progress")
print("  5. stage_2 total is set dynamically after stage_1 completes")

In [ ]:
#@title 🎧 Listen: Visualizations
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_12_visualizations.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 7: Visualization -- Provenance and Scratchpad Metrics

In [ ]:
# ============================================================
# 7A: Claim verification status distribution
# ============================================================

# Process all 50 documents to get full statistics
scratchpad_full = create_scratchpad("full-batch", 50)
all_claims_data = []

for i in range(50):
    doc_id = f"doc_{i+1:03d}"
    result = simulate_document_extraction(doc_id, i)
    scratchpad_full = update_scratchpad(scratchpad_full, result)

    for claim in result.get("claims", []):
        has_source = claim.get("has_source_quote", False)
        conf = claim.get("confidence", 0)

        if not has_source:
            status = "flagged"
        elif conf < 0.7:
            status = "low_confidence"
        else:
            status = "verified"

        all_claims_data.append({
            "doc_id": doc_id,
            "status": status,
            "confidence": conf,
        })

# Count statuses
status_counts = defaultdict(int)
for c in all_claims_data:
    status_counts[c["status"]] += 1

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Verification status pie chart
ax = axes[0]
labels = list(status_counts.keys())
sizes = list(status_counts.values())
colors_map = {
    "verified": "#51cf66",
    "low_confidence": "#ffd43b",
    "flagged": "#ff6b6b",
}
colors = [colors_map.get(l, "#999") for l in labels]
ax.pie(sizes, labels=[f"{l}\n({s})" for l, s in zip(labels, sizes)],
       colors=colors, autopct="%1.1f%%", startangle=90,
       textprops={"fontsize": 10, "fontweight": "bold"})
ax.set_title("Claim Verification Status\n(50 documents processed)",
             fontsize=13, fontweight="bold")

# Panel 2: Confidence distribution histogram
ax = axes[1]
confidences = [c["confidence"] for c in all_claims_data]
ax.hist(confidences, bins=20, color="#4c6ef5", edgecolor="white",
        linewidth=1, alpha=0.85)
ax.axvline(x=0.7, color="#ff6b6b", linestyle="--", linewidth=2,
           label="Confidence threshold (0.7)")
ax.set_xlabel("Confidence Score", fontsize=12)
ax.set_ylabel("Number of Claims", fontsize=12)
ax.set_title("Confidence Distribution", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)

# Panel 3: Scratchpad growth over documents
ax = axes[2]
doc_numbers = list(range(1, 51))
cumulative_claims = []
running_total = 0
for i in range(50):
    doc_id = f"doc_{i+1:03d}"
    result = simulate_document_extraction(doc_id, i)
    if not result["error"]:
        running_total += len(result["claims"])
    cumulative_claims.append(running_total)

ax.plot(doc_numbers, cumulative_claims, "o-", color="#4c6ef5",
        linewidth=2, markersize=3)
ax.fill_between(doc_numbers, cumulative_claims, alpha=0.15, color="#4c6ef5")
ax.set_xlabel("Documents Processed", fontsize=12)
ax.set_ylabel("Cumulative Claims", fontsize=12)
ax.set_title("Scratchpad Growth\n(Claims Accumulated Over Time)",
             fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Total claims extracted: {len(all_claims_data)}")
print(f"Verified: {status_counts.get('verified', 0)} "
      f"({status_counts.get('verified', 0)/len(all_claims_data)*100:.1f}%)")
print(f"Low confidence: {status_counts.get('low_confidence', 0)} "
      f"({status_counts.get('low_confidence', 0)/len(all_claims_data)*100:.1f}%)")
print(f"Flagged (no source): {status_counts.get('flagged', 0)} "
      f"({status_counts.get('flagged', 0)/len(all_claims_data)*100:.1f}%)")

In [ ]:
# ============================================================
# 7B: Context window usage: with vs without scratchpad
# ============================================================

# Simulate context window usage with and without scratchpads
fig, ax = plt.subplots(figsize=(12, 6))

# Without scratchpad: context grows linearly
# Each document adds ~2000 tokens of results to context
tokens_per_doc = 2000
context_limit = 200000  # 200K token context window
system_prompt_tokens = 5000
current_doc_tokens = 3000

without_scratchpad = []
with_scratchpad = []

for i in range(50):
    # Without scratchpad: all previous results stay in context
    usage_without = system_prompt_tokens + current_doc_tokens + (i * tokens_per_doc)
    without_scratchpad.append(usage_without)

    # With scratchpad: only current doc + stats in context
    # Running stats are ~200 tokens regardless of how many docs processed
    stats_tokens = 200
    usage_with = system_prompt_tokens + current_doc_tokens + stats_tokens
    with_scratchpad.append(usage_with)

doc_numbers = list(range(1, 51))

ax.plot(doc_numbers, without_scratchpad, "o-", color="#ff6b6b",
        linewidth=2.5, markersize=4, label="Without Scratchpad")
ax.plot(doc_numbers, with_scratchpad, "o-", color="#51cf66",
        linewidth=2.5, markersize=4, label="With Scratchpad")
ax.axhline(y=context_limit, color="gray", linestyle="--", linewidth=2,
           alpha=0.7, label=f"Context Limit ({context_limit:,} tokens)")

# Mark where context overflows
overflow_doc = None
for i, usage in enumerate(without_scratchpad):
    if usage > context_limit:
        overflow_doc = i + 1
        break

if overflow_doc:
    ax.axvline(x=overflow_doc, color="#ff6b6b", linestyle=":",
               alpha=0.5)
    ax.annotate(f"Context overflow\nat doc {overflow_doc}!",
                xy=(overflow_doc, context_limit),
                xytext=(overflow_doc + 5, context_limit * 0.85),
                fontsize=11, fontweight="bold", color="#ff6b6b",
                arrowprops=dict(arrowstyle="->", color="#ff6b6b"))

ax.set_xlabel("Documents Processed", fontsize=13)
ax.set_ylabel("Context Window Usage (tokens)", fontsize=13)
ax.set_title("Context Window Usage: With vs Without Scratchpad",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=11, loc="upper left")
ax.set_ylim(0, context_limit * 1.15)
ax.yaxis.set_major_formatter(plt.FuncFormatter(
    lambda x, p: f"{x/1000:.0f}K"
))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Without scratchpad: context overflows at document {overflow_doc}")
print(f"With scratchpad: context stays flat at ~{with_scratchpad[0]:,} tokens")
print(f"\nThe scratchpad keeps the context window lean by offloading")
print(f"accumulated results to persistent storage. Only the current")
print(f"document and running statistics stay in context.")

In [ ]:
# ============================================================
# 7C: Temporal validity overview
# ============================================================

# Simulate claims with varying document ages
ages_days = [random.randint(0, 800) for _ in range(200)]
statuses = []
for age in ages_days:
    if age > 365:
        statuses.append("outdated")
    elif age > 270:
        statuses.append("aging")
    else:
        statuses.append("fresh")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Document age distribution
ax = axes[0]
colors_age = ["#51cf66" if s == "fresh" else "#ffd43b" if s == "aging"
              else "#ff6b6b" for s in statuses]
ax.scatter(range(len(ages_days)), ages_days, c=colors_age, alpha=0.6, s=30)
ax.axhline(y=365, color="#ff6b6b", linestyle="--", linewidth=2,
           label="Outdated threshold (365 days)")
ax.axhline(y=270, color="#ffd43b", linestyle="--", linewidth=1.5,
           label="Aging threshold (270 days)")
ax.set_xlabel("Claim Index", fontsize=12)
ax.set_ylabel("Source Document Age (days)", fontsize=12)
ax.set_title("Source Document Age by Claim", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="upper left")

# Panel 2: Freshness breakdown
ax = axes[1]
freshness_counts = defaultdict(int)
for s in statuses:
    freshness_counts[s] += 1

labels = ["Fresh\n(<270 days)", "Aging\n(270-365)", "Outdated\n(>365 days)"]
sizes = [freshness_counts["fresh"], freshness_counts["aging"],
         freshness_counts["outdated"]]
colors = ["#51cf66", "#ffd43b", "#ff6b6b"]

bars = ax.bar(labels, sizes, color=colors, edgecolor="white", linewidth=1.5)
for bar, count in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            str(count), ha="center", fontsize=13, fontweight="bold")

ax.set_ylabel("Number of Claims", fontsize=12)
ax.set_title("Temporal Validity Breakdown", fontsize=13, fontweight="bold")
ax.set_ylim(0, max(sizes) * 1.2)

plt.tight_layout()
plt.show()

print(f"Fresh claims: {freshness_counts['fresh']} "
      f"({freshness_counts['fresh']/200*100:.0f}%)")
print(f"Aging claims: {freshness_counts['aging']} "
      f"({freshness_counts['aging']/200*100:.0f}%)")
print(f"Outdated claims: {freshness_counts['outdated']} "
      f"({freshness_counts['outdated']/200*100:.0f}%)")
print(f"\nOutdated claims should be re-extracted from current documents.")
print(f"Aging claims should be scheduled for re-extraction soon.")

In [ ]:
#@title 🎧 Listen: Mini Project
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_13_mini_project.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 8: Mini-Project -- Full Document Processing Pipeline with Provenance and Scratchpad

Let's bring everything together: claim-source mappings, temporal tracking, conflict detection, and scratchpad-based processing in one integrated pipeline.

In [ ]:
# ============================================================
# 8A: The integrated pipeline
# ============================================================

class DocumentProcessingPipeline:
    """
    End-to-end pipeline for processing multiple documents with
    full provenance tracking and scratchpad-based state management.
    """

    def __init__(self, task_id: str, document_ids: list[str]):
        self.task_id = task_id
        self.document_ids = document_ids
        self.tracker = ProvenanceTrackerSolution(task_id)
        self.scratchpad = create_scratchpad(task_id, len(document_ids))
        self.claim_counter = 0

    def process_document(self, doc_id: str, doc_index: int) -> dict:
        """Process a single document: extract claims, track provenance."""
        # Simulate extraction
        result = simulate_document_extraction(doc_id, doc_index)

        if result["error"]:
            self.scratchpad["error_log"].append(result["error"])
            self.scratchpad["running_statistics"]["documents_with_errors"] += 1
            return {"status": "error", "doc_id": doc_id, "error": result["error"]}

        # Convert raw claims to ExtractedClaim objects with provenance
        extracted = []
        for i, raw_claim in enumerate(result["claims"]):
            self.claim_counter += 1
            claim = ExtractedClaim(
                claim_id=f"CLM-{self.claim_counter:04d}",
                claim_text=raw_claim["claim_text"],
                value=raw_claim["value"],
                field_type="contract_term",
                source=SourceReference(
                    document_id=doc_id,
                    document_title=f"Document {doc_id}",
                    page=raw_claim["page"],
                    paragraph=random.randint(1, 10),
                    exact_quote=(
                        f"Simulated quote from {doc_id}, page {raw_claim['page']}"
                        if raw_claim["has_source_quote"] else None
                    ),
                    document_date=NOW - timedelta(
                        days=random.randint(0, 500)
                    ),
                    extraction_timestamp=NOW,
                ),
                confidence=raw_claim["confidence"],
            )
            self.tracker.add_claim(claim)
            extracted.append(claim)

        # Update scratchpad
        self.scratchpad = update_scratchpad(self.scratchpad, result)

        return {
            "status": "success",
            "doc_id": doc_id,
            "claims_extracted": len(extracted),
            "risk_flags": len(result["risk_flags"]),
        }

    def run(self, verbose: bool = True) -> dict:
        """Run the full pipeline across all documents."""
        start_index = self.scratchpad["document_index"]

        if verbose:
            print(f"PROCESSING {len(self.document_ids)} DOCUMENTS:")
            print(f"Starting from index {start_index}")
            print("=" * 65)

        for i in range(start_index, len(self.document_ids)):
            doc_id = self.document_ids[i]
            result = self.process_document(doc_id, i)

            if verbose and (i < 5 or i % 10 == 0 or i == len(self.document_ids) - 1):
                status = result["status"].upper()
                claims = result.get("claims_extracted", 0)
                print(f"  [{i+1:3d}/{len(self.document_ids)}] "
                      f"{doc_id}: {status} | {claims} claims")

        if verbose:
            print(f"\n  Processing complete.")

        return self.report()

    def report(self) -> dict:
        """Generate final pipeline report."""
        audit = self.tracker.audit_report()
        stats = self.scratchpad["running_statistics"]

        return {
            "task_id": self.task_id,
            "documents_processed": self.scratchpad["accumulated_results"]["processed"],
            "total_documents": len(self.document_ids),
            "total_claims": audit["total_claims"],
            "claim_status": audit["status_breakdown"],
            "average_confidence": audit["average_confidence"],
            "conflicts_found": audit["conflicts_found"],
            "temporal_warnings": audit["total_temporal_warnings"],
            "documents_with_errors": stats["documents_with_errors"],
            "flagged_for_review": stats["flagged_for_review"],
            "context_tokens_used": (
                5000 + 3000 + 200  # system + current_doc + stats
            ),
            "context_tokens_saved": (
                stats["total_claims_extracted"] * 2000  # claims NOT in context
            ),
        }


# Run the pipeline
pipeline = DocumentProcessingPipeline(
    task_id="insurance-batch-001",
    document_ids=[f"doc_{i+1:03d}" for i in range(50)],
)

report = pipeline.run(verbose=True)

print("\n\nFINAL PIPELINE REPORT:")
print("=" * 65)
pp(report)

In [ ]:
# ============================================================
# 8B: Pipeline metrics visualization
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Claim status breakdown
ax = axes[0, 0]
status_data = report["claim_status"]
labels = list(status_data.keys())
sizes = list(status_data.values())
colors_status = {
    "verified": "#51cf66",
    "flagged": "#ff6b6b",
    "pending": "#ffd43b",
}
cs = [colors_status.get(l, "#999") for l in labels]
ax.bar(labels, sizes, color=cs, edgecolor="white", linewidth=1.5)
for i, (label, size) in enumerate(zip(labels, sizes)):
    ax.text(i, size + 1, str(size), ha="center", fontsize=12,
            fontweight="bold")
ax.set_title("Claims by Verification Status", fontsize=13,
             fontweight="bold")
ax.set_ylabel("Count", fontsize=12)

# Panel 2: Context savings
ax = axes[0, 1]
used = report["context_tokens_used"]
saved = report["context_tokens_saved"]
bars = ax.bar(["Tokens Used\n(with scratchpad)", "Tokens Saved\n(vs no scratchpad)"],
              [used, saved], color=["#51cf66", "#74c0fc"],
              edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, [used, saved]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
            f"{val:,}", ha="center", fontsize=12, fontweight="bold")
ax.set_title("Context Window Efficiency", fontsize=13, fontweight="bold")
ax.set_ylabel("Tokens", fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(
    lambda x, p: f"{x/1000:.0f}K"
))

# Panel 3: Pipeline reliability metrics
ax = axes[1, 0]
metrics = {
    "Docs Processed": report["documents_processed"],
    "Total Claims": report["total_claims"],
    "Conflicts": report["conflicts_found"],
    "Temporal\nWarnings": report["temporal_warnings"],
    "Errors": report["documents_with_errors"],
    "Flagged": report["claim_status"].get("flagged", 0),
}
metric_names = list(metrics.keys())
metric_values = list(metrics.values())
metric_colors = ["#4c6ef5", "#4c6ef5", "#ff922b", "#ffd43b",
                 "#ff6b6b", "#ff6b6b"]
bars = ax.barh(metric_names, metric_values, color=metric_colors,
               edgecolor="white", linewidth=1.5, height=0.6)
for bar, val in zip(bars, metric_values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=11, fontweight="bold")
ax.set_title("Pipeline Metrics Summary", fontsize=13, fontweight="bold")
ax.set_xlabel("Count", fontsize=12)

# Panel 4: Confidence histogram of final claims
ax = axes[1, 1]
final_confidences = [c.confidence for c in pipeline.tracker.claims]
ax.hist(final_confidences, bins=20, color="#4c6ef5", edgecolor="white",
        alpha=0.85)
ax.axvline(x=0.7, color="#ff6b6b", linestyle="--", linewidth=2,
           label="Threshold (0.7)")
ax.set_xlabel("Confidence Score", fontsize=12)
ax.set_ylabel("Number of Claims", fontsize=12)
ax.set_title("Final Claim Confidence Distribution", fontsize=13,
             fontweight="bold")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Pipeline processed all 50 documents with full provenance tracking,")
print("temporal validation, conflict detection, and scratchpad-based")
print("state management. Every claim is traceable to its source.")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_14_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 9: Key Takeaways

### What You Learned

| # | Concept | Key Insight |
|---|---|---|
| 1 | **Claim-source mapping** | Every extracted claim must link to a specific document, page, and paragraph. No source = flagged for review. |
| 2 | **Temporal validity** | Information has a shelf life. Track document dates and extraction timestamps. Flag outdated sources automatically. |
| 3 | **Conflict detection** | When two sources disagree, make the conflict visible. Prefer the higher-confidence source, but flag for human review. |
| 4 | **Hallucination prevention** | The strongest defense: require the model to cite its source. If it cannot, the claim is flagged. This transforms silent failure into detectable events. |
| 5 | **Scratchpad files** | Offload intermediate results to persistent storage. Keep the context window lean. Save after every step for crash recovery. |
| 6 | **Context vs scratchpad** | If you need it NOW, keep it in context. If you need it LATER, write it to a scratchpad. |

### Decision Framework for Provenance

When designing an extraction system, ask yourself:

1. **Can every claim be traced to a source?** If not, add source references.
2. **Are sources dated?** If not, add temporal metadata.
3. **Can the system detect when sources disagree?** If not, add conflict detection.
4. **Will the context window fill up?** If yes, use scratchpads.
5. **Can the system survive a crash?** If not, add checkpointing.

### When to Use Scratchpads vs Context

| Use Context | Use Scratchpad |
|---|---|
| Current conversation with user | Results from previous research steps |
| Active instructions and rules | Cross-referenced data from multiple sources |
| Immediate task details | Running calculations and intermediate totals |
| Small, focused datasets | Large datasets being processed in chunks |

### Exam Relevance

On the Claude Certified Architect exam, provenance and scratchpad questions typically appear as:
- "How would you ensure every extracted fact is traceable?" (claim-source mappings)
- "This system processes 100 documents. How do you prevent context overflow?" (scratchpad files)
- "Two sources disagree on a term. What should the agent do?" (conflict detection + escalation)
- "The extraction ran 3 months ago. Is the data still valid?" (temporal checks)

### What's Next

You have now completed the core reliability modules of Pod 5. You can apply these patterns together:
- **Context management** (Notebook 1) keeps the model focused
- **Escalation** (Notebook 2) routes hard cases to humans
- **Error propagation** (Notebook 3) handles multi-agent failures
- **Provenance and scratchpads** (this notebook) ensure trust and scalability

These four pillars work together to build AI systems that a business can depend on.

---

*Claude Certified Architect Prep -- Pod 5: Context Management & Reliability*

In [ ]:
# ============================================================
# Final: Section checklist
# ============================================================

print("Notebook 4: Information Provenance & Scratchpads")
print("=" * 55)
print()
print("Section Checklist:")
sections = [
    "1. Why Does This Matter (motivation)",
    "2. Building Intuition (claims without vs with provenance)",
    "3. Core Implementation (claim-source mappings + temporal checks)",
    "4. Exercise: Provenance Tracker + Claim Verification",
    "5. Scratchpad Files (schema + read-write-checkpoint pattern)",
    "6. Exercise: Multi-Stage Scratchpad Design",
    "7. Visualization (provenance metrics + context savings)",
    "8. Mini-Project: Full Document Processing Pipeline",
    "9. Key Takeaways + What's Next",
]
for s in sections:
    print(f"  [x] {s}")

print()
print("Key classes/functions defined:")
functions = [
    "SourceReference                        -- where a claim came from",
    "ExtractedClaim                         -- claim with source + confidence",
    "ExtractionBatch                        -- batch of claims with summaries",
    "check_temporal_validity()              -- flag outdated sources",
    "detect_conflicts()                     -- find value disagreements",
    "ProvenanceTrackerSolution              -- full provenance tracking system",
    "verify_claim_against_source_solution() -- verify quotes in documents",
    "create_scratchpad()                    -- scratchpad schema creator",
    "update_scratchpad()                    -- write step of R-W-C pattern",
    "DocumentProcessingPipeline             -- integrated end-to-end system",
]
for f in functions:
    print(f"  - {f}")

print()
print("All code is self-contained and runnable without an API key.")
print("You have completed all four reliability notebooks for Pod 5.")